# KIT suite profile

**Goal:** extract the distributions the warehouse generator needs (T18, T19) from the KIT Manual Warehouse Order Picking benchmark, so its parameter ranges come from data rather than guesses.

Runs from `data/processed/`. Build that first from the project folder:

```bash
PYTHONPATH=src python -m fulfilment_optimisation.download kit
PYTHONPATH=src python -m fulfilment_optimisation.parsers.kit
```

In [ ]:
import sys

import pandas as pd

from portfolio import ProjectPaths
from portfolio.plotting import set_style

PATHS = ProjectPaths.for_project("01-fulfilment-optimisation")
sys.path.insert(0, str(PATHS.root / "src"))  # the project package, as the tests' conftest does

from fulfilment_optimisation import kit_profile as kp

set_style()

## 1. Load

One row per replication (13,200: 132 settings × 100 shifts). The order-level table (`kit_orders.parquet`, ~12M rows) isn't needed here; T20 uses it.

In [ ]:
instances = pd.read_parquet(PATHS.processed / "kit_instances.parquet")
settings = kp.settings(instances)
print(f"{len(instances):,} replications, {len(settings)} settings")
settings.groupby("design").size()

## 2. Layouts

The LHS settings cover 5-40 aisles and 5-40 locations per aisle independently; OFAT adds four fixed layouts.

In [ ]:
kp.plot_layouts(settings);

## 3. Order volume and arrivals

Order counts per shift are Poisson around each setting's expected count. Without waves, inter-arrival times have CV 1 (exponential).

**Waves don't behave as labelled in the LHS settings.** In the three OFAT wave settings, burstiness rises with the stochasticity weight (CV 1.0 to 2.8). In the 100 LHS settings it doesn't (r = 0.00): CV stays at 1.1-1.5 and tracks order volume instead (r = 0.99). So the generator calibrates wave strength on the OFAT settings and ignores the LHS stochasticity values.

In [ ]:
kp.plot_arrivals(settings);

## 4. Order size

KIT orders are almost all one to three lines, one unit each. That is much smaller than the Henn & Wäscher orders (5-25 items), so the generator must take order size as a parameter.

In [ ]:
kp.plot_order_sizes(instances);

## 5. Due dates

In [ ]:
kp.plot_slack(settings);

## 6. Generator parameter table

Saved by `python -m fulfilment_optimisation.kit_profile` to `data/processed/kit_profile.csv` and `docs/projects/p1-fulfilment-optimisation/results/kit_profile.md`.

In [ ]:
table = kp.profile(instances, kp.repeat_share(PATHS.processed / "kit_orders.parquet"))
pd.set_option("display.max_colwidth", None)
table

## 7. Findings

- **Layout:** 4-40 aisles × 5-40 locations, independent; storage is A-closest-to-depot in every LHS setting, random in most OFAT ones.
- **Volume:** 400-2,000 orders per 8-hour shift, Poisson around the expected count.
- **Arrivals:** a Poisson process, or 5 waves. Wave strength follows the stochasticity weight only in the OFAT settings (CV up to 2.8); in LHS it tracks volume, so LHS stochasticity isn't used for calibration.
- **Order size:** 1-3 lines, mostly 80 / 15 / 5%, one unit per line. Far smaller than Henn & Wäscher orders.
- **Due dates:** shift end, or arrival plus a fixed offset (600 s, 1,800 s, or 4-398 s in LHS).
- **Popularity:** uniform or ABC (20 / 30 / 50% of articles take 80 / 15 / 5% of demand).

**For T18/T19:** sample layout size, volume and due-date offset from these ranges, with wave strength from the OFAT settings; take the order-size distribution as a parameter (KIT-like 1-3 lines, or H&W-like 5-25 items) so the batching benchmarks stay comparable.